# 🚀 RupRao (รูปเรา) — สแกนคลังรูปภาพกิจกรรมด้วย GPU บน Google Colab

> สมุดโค้ดสำหรับสแกนคลังภาพขนาดใหญ่จาก Google Drive อัตโนมัติ:
- ใช้ GPU ฟรี (Tesla T4) ของ Google Colab รัน InsightFace เร็วกว่า CPU 15–20 เท่า
- บันทึกเมทาดาทาลง SQLite () และเวกเตอร์ 512 มิติลง FAISS ()
- มีระบบ **Resume** อัตโนมัติ: หากหลุดหรือหมดเวลา Colab สามารถกดรันต่อจากเดิมได้ทันที ไม่เริ่มใหม่

### 1️⃣ ตรวจสอบการเชื่อมต่อ GPU

In [ ]:
!nvidia-smi
import torch
print("
🎮 GPU พร้อมใช้งาน:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("ชื่อ GPU:", torch.cuda.get_device_name(0))

### 2️⃣ เชื่อมต่อ Google Drive (ที่เก็บรูปภาพกิจกรรม)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

### 3️⃣ ติดตั้งไลบรารี AI และสร้างสภาพแวดล้อม

In [ ]:
# ติดตั้ง InsightFace, ONNX Runtime GPU, FAISS, Pillow
!pip install -q "insightface==1.0.1" "onnxruntime-gpu" "faiss-cpu" "pillow" "tqdm"
print("✅ ติดตั้งไลบรารีเสร็จสมบูรณ์!")

### 4️⃣ กำหนดโฟลเดอร์รูปภาพบน Google Drive

In [ ]:
import os
from pathlib import Path

# ปรับเส้นทางให้ตรงกับโฟลเดอร์บน Google Drive ของคุณ
DRIVE_ROOT = Path("/content/drive/MyDrive/RupRao")
PHOTOS_DIR = DRIVE_ROOT / "photos"          # โฟลเดอร์เก็บรูปภาพงานทั้งหมด
OUTPUT_DIR = DRIVE_ROOT / "output"          # โฟลเดอร์เก็บผลลัพธ์ (db + index + thumbs)
DB_PATH = OUTPUT_DIR / "faces.db"
INDEX_PATH = OUTPUT_DIR / "index.faiss"
THUMB_DIR = OUTPUT_DIR / "thumbs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
THUMB_DIR.mkdir(parents=True, exist_ok=True)
print("📁 ตั้งค่าเส้นทางเรียบร้อย:")
print("  - โฟลเดอร์รูปภาพ :", PHOTOS_DIR)
print("  - โฟลเดอร์ผลลัพธ์:", OUTPUT_DIR)

### 5️⃣ โค้ดระบบสแกนคลังภาพ (Batch GPU + Resume)

In [ ]:
import sqlite3
import cv2
import faiss
import numpy as np
from PIL import Image, ImageOps
from tqdm import tqdm
from insightface.app import FaceAnalysis

# 1. โหลดโมเดล AI บน GPU (CUDA)
print("🧠 โหลดโมเดล InsightFace buffalo_l บน GPU...")
app = FaceAnalysis(name="buffalo_l", providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
app.prepare(ctx_id=0, det_size=(640, 640))
print("✅ โมเดล AI บน GPU พร้อมทำงาน!
")

# 2. ฟังก์ชันฐานข้อมูล SQLite
def init_db(db_path):
    conn = sqlite3.connect(str(db_path))
    conn.execute("PRAGMA foreign_keys = ON;")
    with conn:
        conn.executescript("""
        CREATE TABLE IF NOT EXISTS photos (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            rel_path TEXT UNIQUE NOT NULL,
            event_name TEXT NOT NULL,
            width INTEGER,
            height INTEGER,
            file_size INTEGER,
            scanned_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        );
        CREATE TABLE IF NOT EXISTS faces (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            photo_id INTEGER NOT NULL,
            face_index INTEGER NOT NULL,
            bbox_x1 INTEGER NOT NULL,
            bbox_y1 INTEGER NOT NULL,
            bbox_x2 INTEGER NOT NULL,
            bbox_y2 INTEGER NOT NULL,
            det_score REAL NOT NULL,
            faiss_id INTEGER UNIQUE NOT NULL,
            FOREIGN KEY (photo_id) REFERENCES photos (id) ON DELETE CASCADE
        );
        CREATE INDEX IF NOT EXISTS idx_photos_rel ON photos(rel_path);
        CREATE INDEX IF NOT EXISTS idx_faces_pid ON faces(photo_id);
        CREATE INDEX IF NOT EXISTS idx_faces_fid ON faces(faiss_id);
        """)
    return conn

# 3. โหลดหรือสร้าง FAISS Index
def get_faiss_index(index_path):
    if index_path.exists():
        print(f"📂 โหลด FAISS Index เดิมจาก {index_path}")
        return faiss.read_index(str(index_path))
    base = faiss.IndexFlatIP(512)
    return faiss.IndexIDMap2(base)

# 4. ฟังก์ชันย่อภาพ WebP
def make_thumb(src_path, dest_path):
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    if dest_path.exists(): return
    with Image.open(src_path) as img:
        img = ImageOps.exif_transpose(img)
        img.thumbnail((640, 640), Image.Resampling.LANCZOS)
        img.save(dest_path, "WEBP", quality=80)
print("✅ ฟังก์ชันระบบพร้อมทำงาน!")

### 6️⃣ รันการสแกนคลังรูปภาพทั้งหมด (Progress Tracking + Auto-Save)

In [ ]:
conn = init_db(DB_PATH)
index = get_faiss_index(INDEX_PATH)

# ดึงรายชื่อรูปที่สแกนแล้ว (ระบบ Resume)
cur = conn.cursor()
cur.execute("SELECT rel_path FROM photos")
scanned_set = {row[0] for row in cur.fetchall()}
print(f"📊 รูปที่สแกนเสร็จไปแล้วก่อนหน้านี้: {len(scanned_set)} รูป")

# ค้นหารูปทั้งหมดในโฟลเดอร์
valid_exts = {".jpg", ".jpeg", ".png", ".webp", ".JPG", ".JPEG", ".PNG"}
all_files = sorted([p for p in PHOTOS_DIR.rglob("*") if p.suffix in valid_exts])
print(f"🔍 พบไฟล์รูปภาพทั้งหมดใน Drive: {len(all_files)} รูป
")

# หา faiss_id ล่าสุด
cur.execute("SELECT COALESCE(MAX(faiss_id), -1) + 1 FROM faces")
next_faiss_id = int(cur.fetchone()[0])

newly_scanned = 0
new_faces_count = 0

for file_path in tqdm(all_files, desc="🚀 สแกนภาพด้วย GPU"):
    try:
        rel_path = str(file_path.relative_to(PHOTOS_DIR))
    except Exception:
        rel_path = file_path.name

    # ข้ามรูปที่ทำเสร็จแล้ว
    if rel_path in scanned_set:
        continue

    try:
        img = cv2.imread(str(file_path))
        if img is None: continue
        h, w = img.shape[:2]
        file_size = file_path.stat().st_size
        event_name = file_path.parent.name if file_path.parent != PHOTOS_DIR else "กิจกรรมทั่วไป"

        faces = app.get(img)
        cur.execute(
            "INSERT INTO photos (rel_path, event_name, width, height, file_size) VALUES (?, ?, ?, ?, ?)",
            (rel_path, event_name, w, h, file_size)
        )
        photo_id = cur.lastrowid

        for idx, face in enumerate(faces):
            if float(face.det_score) < 0.45: continue
            emb = np.asarray(face.embedding, dtype=np.float32)
            norm = np.linalg.norm(emb)
            if norm <= 0: continue
            emb = (emb / norm).reshape(1, -1)

            # บันทึกลง FAISS
            index.add_with_ids(emb, np.array([next_faiss_id], dtype=np.int64))

            # บันทึกลง SQLite
            x1, y1, x2, y2 = [int(v) for v in face.bbox]
            cur.execute(
                """INSERT INTO faces (photo_id, face_index, bbox_x1, bbox_y1, bbox_x2, bbox_y2, det_score, faiss_id)
                   VALUES (?, ?, ?, ?, ?, ?, ?, ?)""",
                (photo_id, idx, x1, y1, x2, y2, float(face.det_score), next_faiss_id)
            )
            next_faiss_id += 1
            new_faces_count += 1

        # สร้าง Thumbnail
        thumb_file = (THUMB_DIR / rel_path).with_suffix(".webp")
        make_thumb(file_path, thumb_file)

        newly_scanned += 1
        scanned_set.add(rel_path)

        # Auto-save ทุกๆ 50 รูป (ปลอดภัยแม้หมดเวลา Colab)
        if newly_scanned % 50 == 0:
            conn.commit()
            faiss.write_index(index, str(INDEX_PATH))

    except Exception as e:
        print(f"⚠️ เกิดข้อผิดพลาดกับรูป {rel_path}: {e}")

# บันทึกรอบสุดท้าย
conn.commit()
faiss.write_index(index, str(INDEX_PATH))

print("
🎉 สแกนเสร็จสิ้นเรียบร้อย!")
print(f"- รูปที่สแกนรอบนี้ : {newly_scanned:,} รูป")
print(f"- หน้าที่พบใหม่    : {new_faces_count:,} หน้า")
print(f"- รวมเวกเตอร์ใน FAISS: {index.ntotal:,} รายการ")

### 7️⃣ วิธีนำผลลัพธ์กลับมาใช้บนเซิร์ฟเวอร์ RupRao บน Mac

1. ดาวน์โหลดโฟลเดอร์  จาก Google Drive ลงมาที่เครื่อง Mac
2. ก๊อปปี้ไฟล์  และ  วางลงในโฟลเดอร์ 
3. ก๊อปปี้โฟลเดอร์  วางลงใน 
4. สั่งรัน  หรือ  ค้นหาได้ทันทีในเสี้ยววินาที!